# Camada Silver - Women's Clothing Reviews + Rent the Runway + US Holiday Dates

Lê as tabelas Bronze, aplica as soluções definidas no `artefacto_quality_womens_rtr.xlsx` e grava as tabelas Silver (Delta).
O Rent the Runway e os feriados ficam só com o intervalo de tempo comum aos datasets do grupo.

| Dataset | Tabela Silver | Uma linha por |
|---|---|---|
| Women's Clothing | silver.avaliacoes | avaliação |
| Women's Clothing | silver.produtos | produto e divisão |
| Rent the Runway | silver.avaliacoes_rtr | avaliação (junho a dezembro de 2017) |
| Rent the Runway | silver.produtos_rtr | peça |
| US Holiday Dates | silver.feriados | feriado (junho a dezembro de 2017) |

In [1]:
# Intervalo de tempo comum aos datasets do grupo (alterar aqui se o grupo mudar o intervalo)
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta import *

# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .appName("Silver - Womens Clothing + Rent the Runway + US Holidays") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

## 1. Women's E-Commerce Clothing Reviews

Tratamentos (folha `Womens_Clothing_Reviews` do Excel):
- **Unnamed: 0**: eliminado; remove-se as 21 linhas duplicadas nas restantes colunas e gera-se a chave própria `id_avaliacao`.
- **Clothing ID = 0**: verificado; tem a classificação completa (General / Jackets / Outerwear), por isso é um produto real e mantém-se.
- **Age, Title, Review Text, Positive Feedback Count**: eliminados (não são usados nas questões).
- **Rating**: inteiro; cria-se `avaliacao_negativa` (rating ≤ 2).
- **Recommended IND**: convertido para booleano `recomenda`; cria-se `avaliacao_incoerente`.
- **Division Name**: "Initmates" → "Intimates"; nulos → "Desconhecido".
- **Department Name / Class Name**: nulos → "Desconhecido"; classes com menos de 10 avaliações → "Outros"; a classe do Clothing ID 1119 é uniformizada para "Outerwear".
- O dataset não tem datas, por isso não é afetado pelo intervalo de tempo do grupo.

In [4]:
wc = spark.table("bronze.bronze_reviews")

colunas_dados = [c for c in wc.columns if c not in ("Unnamed: 0", "_data_ingestao", "_ficheiro_origem")]
wc = wc.withColumn("indice_origem", F.col("`Unnamed: 0`").cast("int"))

# remover duplicados (mantém a 1.ª ocorrência no ficheiro)
w_dup = Window.partitionBy(*[F.col(f"`{c}`") for c in colunas_dados]).orderBy("indice_origem")
antes = wc.count()
wc = wc.withColumn("_n", F.row_number().over(w_dup)).filter("_n = 1").drop("_n")
print("duplicados removidos:", antes - wc.count())

wc = wc.select(
    F.row_number().over(Window.orderBy("indice_origem")).alias("id_avaliacao"),
    F.col("`Clothing ID`").cast("int").alias("clothing_id"),
    F.col("Rating").cast("int").alias("rating"),
    (F.col("`Recommended IND`").cast("int") == 1).alias("recomenda"),
    F.coalesce(F.regexp_replace("`Division Name`", "^Initmates$", "Intimates"), F.lit("Desconhecido")).alias("division_name"),
    F.coalesce(F.col("`Department Name`"), F.lit("Desconhecido")).alias("department_name"),
    F.coalesce(F.col("`Class Name`"), F.lit("Desconhecido")).alias("class_name"),
)

# classe do Clothing ID 1119 (Outerwear 6x, Jackets 2x) e classes raras
wc = wc.withColumn("class_name", F.when(F.col("clothing_id") == 1119, "Outerwear").otherwise(F.col("class_name")))
classes_raras = [r["class_name"] for r in wc.groupBy("class_name").count().filter("count < 10").collect()]
print("classes agrupadas em 'Outros':", classes_raras)
wc = wc.withColumn("class_name", F.when(F.col("class_name").isin(classes_raras), "Outros").otherwise(F.col("class_name")))

wc = wc \
    .withColumn("avaliacao_negativa", F.col("rating") <= 2) \
    .withColumn("avaliacao_incoerente",
                ((F.col("rating") >= 4) & ~F.col("recomenda")) | ((F.col("rating") <= 2) & F.col("recomenda")))
wc.cache()

avaliacoes = wc.select("id_avaliacao", "clothing_id", "division_name", "rating", "recomenda",
                       "avaliacao_negativa", "avaliacao_incoerente")
produtos = wc.select("clothing_id", "division_name", "department_name", "class_name").distinct()

# a chave (clothing_id, division_name) tem de ser única
assert produtos.count() == produtos.select("clothing_id", "division_name").distinct().count()

gravar_silver(avaliacoes, "avaliacoes")
gravar_silver(produtos, "produtos")

duplicados removidos: 21
classes agrupadas em 'Outros': ['Chemises', 'Casual bottoms']
silver.avaliacoes: 23465 linhas
silver.produtos: 1402 linhas


## 2. Rent the Runway

Tratamentos (folha `Rent_the_Runway` do Excel):
- Remove-se as 189 linhas duplicadas e as avaliações sem nota.
- **review_date**: convertida de texto ("April 20, 2016") para data; ficam só as avaliações do intervalo do grupo.
- **rating**: convertido para inteiro (2 a 10); cria-se `nota_5` (rating / 2) e `avaliacao_negativa` (rating ≤ 4, ou seja ≤ 2 estrelas).
- **fit**: renomeado para `ajuste`.
- **category**: agrupada nos departamentos do Women's Clothing (`departamento_wc`); as 5 peças com duas categorias ficam com a mais frequente.
- Os atributos da cliente (medidas, idade, tipo de corpo), o texto e a ocasião do aluguer são eliminados.

In [5]:
rtr = spark.table("bronze.bronze_rtr_reviews")
colunas_rtr = [c for c in rtr.columns if c not in ("_data_ingestao", "_ficheiro_origem")]

antes = rtr.count()
rtr = rtr.dropDuplicates(colunas_rtr)
print("duplicados removidos:", antes - rtr.count())

rtr = rtr \
    .filter(F.col("rating").isNotNull()) \
    .withColumn("data_avaliacao", F.to_date("review_date", "MMMM d, yyyy")) \
    .withColumn("nota", F.col("rating").cast("int")) \
    .withColumn("item_id", F.col("item_id").cast("int"))

assert rtr.filter(F.col("data_avaliacao").isNull()).count() == 0, "datas por converter"
rtr = rtr.filter(F.col("data_avaliacao").between(DATA_INICIO, DATA_FIM))
rtr.cache()
print("avaliações no intervalo:", rtr.count())

duplicados removidos: 189
avaliações no intervalo: 43643


In [6]:
avaliacoes_rtr = rtr.select(
        F.row_number().over(Window.orderBy("data_avaliacao", "user_id", "item_id")).alias("id_avaliacao_rtr"),
        "item_id",
        "data_avaliacao",
        "nota",
        (F.col("nota") / 2).alias("nota_5"),
        (F.col("nota") <= 4).alias("avaliacao_negativa"),
        F.col("fit").alias("ajuste"))

gravar_silver(avaliacoes_rtr, "avaliacoes_rtr")

silver.avaliacoes_rtr: 43643 linhas


In [7]:
# categorias do Rent the Runway -> departamentos do Women's Clothing
departamentos = {
    "Dresses": ["dress", "gown", "sheath", "shift", "maxi", "mini", "shirtdress", "midi", "frock", "ballgown", "print", "caftan", "kaftan"],
    "Tops": ["top", "blouse", "sweater", "shirt", "tee", "t-shirt", "pullover", "tank", "sweatshirt", "cardigan", "turtleneck", "knit",
             "cami", "henley", "tunic", "blouson", "hoodie", "sweatershirt", "kimono", "buttondown"],
    "Bottoms": ["skirt", "skirts", "pants", "pant", "jeans", "culottes", "culotte", "leggings", "legging", "trousers", "trouser",
                "sweatpants", "jogger", "overalls", "skort"],
    "Jackets": ["jacket", "coat", "blazer", "bomber", "vest", "trench", "parka", "peacoat", "cape", "poncho", "overcoat", "down", "duster"],
}
mapa = spark.createDataFrame([(c, d) for d, cs in departamentos.items() for c in cs], ["categoria", "departamento_wc"])

# categoria mais frequente de cada peça (5 peças aparecem com duas categorias)
w_cat = Window.partitionBy("item_id").orderBy(F.desc("n"), "categoria")
produtos_rtr = rtr.groupBy("item_id", F.col("category").alias("categoria")).agg(F.count("*").alias("n")) \
    .withColumn("_r", F.row_number().over(w_cat)).filter("_r = 1") \
    .join(F.broadcast(mapa), "categoria", "left") \
    .withColumn("departamento_wc", F.coalesce("departamento_wc", F.lit("Outros"))) \
    .select("item_id", "categoria", "departamento_wc")

assert produtos_rtr.count() == produtos_rtr.select("item_id").distinct().count()
gravar_silver(produtos_rtr, "produtos_rtr")
produtos_rtr.groupBy("departamento_wc").count().orderBy(F.desc("count")).show()

silver.produtos_rtr: 5475 linhas
+---------------+-----+
|departamento_wc|count|
+---------------+-----+
|        Dresses| 3627|
|           Tops|  861|
|        Jackets|  470|
|         Outros|  290|
|        Bottoms|  227|
+---------------+-----+



## 3. US Holiday Dates

Tratamentos (folha `US_Holiday_Dates` do Excel):
- **Date**: convertida para data; ficam só os feriados do intervalo do grupo.
- **Holiday**: uniformiza-se o apóstrofo ("New Year’s Eve" → "New Year's Eve").
- **WeekDay, Month, Day, Year**: eliminados (derivam da data).

In [8]:
feriados = spark.table("bronze.bronze_us_holidays") \
    .select(F.to_date("Date").alias("data"),
            F.regexp_replace("Holiday", "\u2019", "'").alias("feriado")) \
    .filter(F.col("data").between(DATA_INICIO, DATA_FIM)) \
    .dropDuplicates()

gravar_silver(feriados, "feriados")
feriados.orderBy("data").show(truncate=False)

silver.feriados: 12 linhas
+----------+-----------------+
|data      |feriado          |
+----------+-----------------+
|2017-06-19|Juneteenth       |
|2017-07-04|4th of July      |
|2017-09-02|Labor Day Weekend|
|2017-09-03|Labor Day Weekend|
|2017-09-04|Labor Day        |
|2017-10-09|Columbus Day     |
|2017-11-11|Veterans Day     |
|2017-11-22|Thanksgiving Eve |
|2017-11-23|Thanksgiving Day |
|2017-12-24|Christmas Eve    |
|2017-12-25|Christmas Day    |
|2017-12-31|New Year's Eve   |
+----------+-----------------+



## 4. Verificação

Confirma que as tabelas Silver reproduzem os valores atuais dos KPIs indicados no relatório.

In [9]:
spark.sql("SHOW TABLES FROM silver").show(truncate=False)

# KPI 1 (Women's Clothing): taxa de recomendação global e por departamento
spark.sql("""
    SELECT ROUND(100 * AVG(CAST(recomenda AS INT)), 1) AS taxa_recomendacao,
           SUM(CAST(avaliacao_incoerente AS INT))      AS incoerentes
    FROM silver.avaliacoes
""").show()

spark.sql("""
    SELECT p.department_name, COUNT(*) AS avaliacoes, ROUND(100 * AVG(CAST(a.recomenda AS INT)), 1) AS taxa_recomendacao
    FROM silver.avaliacoes a
    JOIN silver.produtos p ON p.clothing_id = a.clothing_id AND p.division_name = a.division_name
    GROUP BY p.department_name
    ORDER BY taxa_recomendacao
""").show()

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|silver   |avaliacoes      |false      |
|silver   |avaliacoes_rtr  |false      |
|silver   |feriados        |false      |
|silver   |marketing_funnel|false      |
|silver   |produtos        |false      |
|silver   |produtos_rtr    |false      |
+---------+----------------+-----------+

+-----------------+-----------+
|taxa_recomendacao|incoerentes|
+-----------------+-----------+
|             82.2|        303|
+-----------------+-----------+

+---------------+----------+-----------------+
|department_name|avaliacoes|taxa_recomendacao|
+---------------+----------+-----------------+
|          Trend|       119|             73.9|
|        Dresses|      6312|             80.8|
|           Tops|     10455|             81.5|
|        Jackets|      1032|             83.6|
|       Intimate|      1735|             85.0|
|        Bottoms|      3798|             85.1|
|   D

In [10]:
# KPI 2: % de avaliações negativas por departamento (Rent the Runway) e recomendação no Women's Clothing
spark.sql("""
    WITH rtr AS (
        SELECT p.departamento_wc AS departamento,
               COUNT(*) AS avaliacoes_rtr,
               ROUND(100 * AVG(CAST(a.avaliacao_negativa AS INT)), 1) AS pct_negativas_rtr
        FROM silver.avaliacoes_rtr a
        JOIN silver.produtos_rtr p ON p.item_id = a.item_id
        GROUP BY p.departamento_wc),
    wc AS (
        SELECT p.department_name AS departamento,
               ROUND(100 * AVG(CAST(a.recomenda AS INT)), 1) AS taxa_recomendacao_wc
        FROM silver.avaliacoes a
        JOIN silver.produtos p ON p.clothing_id = a.clothing_id AND p.division_name = a.division_name
        GROUP BY p.department_name)
    SELECT rtr.*, wc.taxa_recomendacao_wc
    FROM rtr LEFT JOIN wc ON wc.departamento = rtr.departamento
    ORDER BY pct_negativas_rtr DESC
""").show()

spark.sql("""
    SELECT COUNT(*) AS avaliacoes, ROUND(100 * AVG(CAST(avaliacao_negativa AS INT)), 1) AS pct_negativas_global
    FROM silver.avaliacoes_rtr
""").show()

+------------+--------------+-----------------+--------------------+
|departamento|avaliacoes_rtr|pct_negativas_rtr|taxa_recomendacao_wc|
+------------+--------------+-----------------+--------------------+
|        Tops|          3930|              3.6|                81.5|
|      Outros|          2006|              2.9|                null|
|     Bottoms|          1153|              2.9|                85.1|
|     Jackets|          1475|              2.7|                83.6|
|     Dresses|         35079|              1.9|                80.8|
+------------+--------------+-----------------+--------------------+

+----------+--------------------+
|avaliacoes|pct_negativas_global|
+----------+--------------------+
|     43643|                 2.2|
+----------+--------------------+



In [11]:
# KPI 3: semanas (segunda a domingo) com pelo menos um feriado vs restantes semanas
spark.sql("""
    WITH semanas_feriado AS (SELECT DISTINCT date_trunc('week', data) AS semana FROM silver.feriados)
    SELECT CASE WHEN s.semana IS NOT NULL THEN 'semana com feriado' ELSE 'restantes semanas' END AS tipo,
           COUNT(*) AS avaliacoes,
           ROUND(100 * AVG(CAST(a.avaliacao_negativa AS INT)), 2) AS pct_negativas
    FROM silver.avaliacoes_rtr a
    LEFT JOIN semanas_feriado s ON s.semana = date_trunc('week', a.data_avaliacao)
    GROUP BY 1
""").show()

+------------------+----------+-------------+
|              tipo|avaliacoes|pct_negativas|
+------------------+----------+-------------+
|semana com feriado|     12196|         1.94|
| restantes semanas|     31447|         2.25|
+------------------+----------+-------------+



In [12]:
spark.stop()